# 贝叶斯网络作业：精确推理与近似推理

本次编程作业基于课程讲解的贝叶斯网络（Bayesian Network），旨在对贝叶斯网络精确推理和近似推理的基本方法有更深的掌握，要求实现**枚举推理（Enumeration Inference）**和**拒绝采样（Rejection Sampling）**。

| 部分 | 内容 | 待完成项 |
|------|------|----------|
| 部分 0 | 贝叶斯网络定义（已给出） | 0 |
| 部分 1 | 局部条件概率计算 `get_prob` | 1 |
| 部分 2 | 枚举推理：`enumerate_all` | 2 |
| 部分 3 | 拒绝采样：`prior_sample` + `rejection_sampling` | 2 |
| 部分 4 | 验证（已给出，直接运行） | 0 |

> 完成全部 TODO，并确保每个检查单元在提交前都打印 OK。


---
## 部分 0：网络定义

这里我们使用经典的 **Burglary-Earthquake-Alarm** 网络。无需修改，直接运行，该网络取自教材，可以与教材联动学习。


In [1]:
import random

# 定义贝叶斯网络结构和条件概率表 (CPTs)
BURGLARY_NET = {
    'Variables': ['B', 'E', 'A', 'J', 'M'],
    'Parents': {
        'B': [],
        'E': [],
        'A': ['B', 'E'],
        'J': ['A'],
        'M': ['A']
    },
    'CPT': {
        # 格式: { (父节点取值元组): 变量为 True 的概率 }
        'B': {(): 0.001},
        'E': {(): 0.002},
        'A': {
            (True, True): 0.95,
            (True, False): 0.94,
            (False, True): 0.29,
            (False, False): 0.001
        },
        'J': {
            (True,): 0.90,
            (False,): 0.05
        },
        'M': {
            (True,): 0.70,
            (False,): 0.01
        }
    }
}

print('Part 0 loaded OK')
print('Variables:', BURGLARY_NET['Variables'])


Part 0 loaded OK
Variables: ['B', 'E', 'A', 'J', 'M']


---
## 部分 1：概率查询

### TODO 1 / 5 -- `get_prob`

给定变量 `var`、候选真值 `val` (True/False) 以及当前证据字典 `e`，从网络的 CPT 中提取 $P(var=val | parents(var))$。


In [2]:
def get_prob(var, val, e, bn):
    """
    计算条件概率 P(var = val | parents(var))。

    参数:
        var: 变量名 (str)
        val: 变量的取值 (bool)
        e: 包含父节点取值的字典
        bn: 贝叶斯网络字典
    """
    parents = bn['Parents'][var]
    # 提取父节点的取值，构成元组作为 CPT 字典的键
    parent_vals = tuple(e[p] for p in parents)
    
    # ---------------------------------------------------------------
    # TODO 1
    # 从 bn['CPT'][var] 中获取当 var 为 True 时的概率 prob_true。
    # 如果 val 为 True，则返回 prob_true；如果 val 为 False，则返回 1.0 - prob_true。
    # ---------------------------------------------------------------

    # 从 CPT 中获取 var=True 的概率
    prob_true = bn['CPT'][var][parent_vals]
    # 若 val 为 True 返回 prob_true，否则返回 1 - prob_true
    if val:
        return prob_true
    else:
        return 1.0 - prob_true


In [3]:
# --- 验证 TODO 1 ---
_e = {'B': True, 'E': False, 'A': True}
assert abs(get_prob('B', True, _e, BURGLARY_NET) - 0.001) < 1e-6
assert abs(get_prob('A', True, _e, BURGLARY_NET) - 0.94) < 1e-6
assert abs(get_prob('J', False, _e, BURGLARY_NET) - 0.10) < 1e-6  # 1 - 0.90
print('OK: get_prob')


OK: get_prob


---
## 部分 2：精确推理 (Enumeration)

根据全概率公式和链式法则进行精确枚举推理，此题要求计算条件联合概率。

### TODO 2 + 3 / 5 -- `enumerate_all`


In [4]:
def enumerate_all(vars_list, e, bn):
    """
    递归计算联合概率。vars_list 是尚未处理的变量列表。
    """
    if not vars_list:
        return 1.0
    
    Y = vars_list[0]
    rest = vars_list[1:]
    
    if Y in e:
        # ---------------------------------------------------------------
        # TODO 2
        # 如果 Y 的值已经在证据 e 中已知：
        # 返回 P(Y=e[Y] | parents(Y)) 乘以对剩余变量 rest 调用 enumerate_all 的结果。
        # ---------------------------------------------------------------

        # 变量值已知：乘上 P(Y=e[Y] | parents(Y))，然后递归处理剩余变量
        prob = get_prob(Y, e[Y], e, bn)
        return prob * enumerate_all(rest, e, bn)
    else:
        # ---------------------------------------------------------------
        # TODO 3
        # 如果 Y 是隐藏变量：
        # 需要遍历 Y 的所有可能取值（True 和 False），对每种取值：
        #   1. 创建证据字典 e 的副本 e_new，并将 Y 的值设为当前遍历的取值。
        #   2. 计算 P(Y=y | parents(Y)) 乘以 enumerate_all(rest, e_new, bn) 的结果。
        # 返回所有取值结果的累加和。
        # ---------------------------------------------------------------
        total = 0.0
        for y_val in [True, False]:
            e_new = e.copy()
            e_new[Y] = y_val
            prob = get_prob(Y, y_val, e, bn)
            total += prob * enumerate_all(rest, e_new, bn)
        return total
        

def enumeration_ask(X, e, bn):
    """
    计算分布 P(X | e)。返回一个字典，包含 X 为 True 和 False 的概率。
    """
    Q = {}
    for x_val in [True, False]:
        e_extended = e.copy()
        e_extended[X] = x_val
        Q[x_val] = enumerate_all(bn['Variables'], e_extended, bn)
    
    # 归一化
    alpha = sum(Q.values())
    return {k: v / alpha for k, v in Q.items()}


In [5]:
# --- 验证 TODO 2 + 3 ---
# P(Burglary | JohnCalls=True, MaryCalls=True)
_res = enumeration_ask('B', {'J': True, 'M': True}, BURGLARY_NET)
assert abs(_res[True] - 0.284) < 0.01, f"Expected ~0.284, got {_res[True]}"
print('OK: enumeration_ask')


OK: enumeration_ask


---
## 部分 3：近似推理 (Rejection Sampling)

拒绝采样是一种基于马尔可夫链的简单近似推理方法。

### TODO 4 / 5 -- `prior_sample`
按照拓扑顺序无条件地对整个网络进行一次随机采样。


In [6]:
def prior_sample(bn):
    """
    从先验分布中返回一个样本 (字典格式: {var: True/False, ...})
    """
    sample = {}
    # 假设 bn['Variables'] 已经按拓扑顺序排列
    for var in bn['Variables']:
        # ---------------------------------------------------------------
        # TODO 4
        # 使用 get_prob 获取当前 var 为 True 的概率 prob_true。
        # 生成一个随机数 (可用 random.random())，若随机数 < prob_true，
        # 则 sample[var] = True，否则为 False。
        # ---------------------------------------------------------------
        # 计算该变量为 True 的概率
        prob_true = get_prob(var, True, sample, bn)
        # 以 prob_true 的概率赋值为 True，否则为 False
        sample[var] = random.random() < prob_true
    
    return sample


In [7]:
# --- 验证 TODO 4 ---
random.seed(42)
_sample = prior_sample(BURGLARY_NET)
assert len(_sample) == len(BURGLARY_NET['Variables'])
assert type(_sample['B']) == bool
print('OK: prior_sample')


OK: prior_sample


### TODO 5 / 5 -- `rejection_sampling`
拒绝采样算法(rejection sampling)是一种给定一个易于采样的分布，为一个难于采样的分布生成采样样本的通用方法。在其最简单的形式中，它可以被用于计算条件概率。首先，它根据网络指定的先验分布生成采样样本。然后，它拒绝所有与证据不匹配的样本。最后，在剩余样本中通过统计X=x的出现的频次而计算出估计概率

In [8]:
def rejection_sampling(X, e, bn, N):
    """
    使用拒绝采样估算 P(X | e)。
    N: 采样总次数。
    """
    counts = {True: 0, False: 0}
    
    for _ in range(N):
        # ---------------------------------------------------------------
        # TODO 5
        # 1. 调用 prior_sample 生成一个样本 x。
        # 2. 检查 x 是否与证据 e 一致 (即对于 e 中的每个变量，x 中的取值必须与 e 相同)。
        # 3. 如果一致，则将 counts[x[X]] 的计数加 1；如果不一致，则拒绝该样本(什么也不做)。
        # ---------------------------------------------------------------

        # 1. 从先验分布采样
        sample = prior_sample(bn)
        # 2. 检查样本是否与证据一致
        consistent = all(sample[var] == val for var, val in e.items())
        if consistent:
            # 3. 一致则计数
            counts[sample[X]] += 1
        # 不一致则拒绝

    # 归一化
    alpha = sum(counts.values())
    if alpha == 0:  # 避免除以零
        return {True: 0.5, False: 0.5}
    return {k: v / alpha for k, v in counts.items()}


In [9]:
# --- 验证 TODO 5 ---
random.seed(42)
# P(Burglary | Alarm=True)，用枚举法的理论结果约为 0.373
_rej_res = rejection_sampling('B', {'A': True}, BURGLARY_NET, 500000)
assert abs(_rej_res[True] - 0.373) < 0.05, f"Sampling result {_rej_res[True]} too far from 0.373"
print('OK: rejection_sampling')


OK: rejection_sampling


---
## 部分 4：运行与最终检查

运行完整的测试逻辑以确保功能正确。


In [10]:
print("=== 最终测试 ===")
query_X = 'B'
evidence = {'J': True, 'M': True}

print(f"查询: P({query_X} | JohnCalls=True, MaryCalls=True)")

# 精确推理结果
exact_res = enumeration_ask(query_X, evidence, BURGLARY_NET)
print(f"精确推理 (Enumeration) 结果: {exact_res[True]:.4f}")

# 近似推理结果
random.seed(42)
approx_res = rejection_sampling(query_X, evidence, BURGLARY_NET, 500000)
print(f"近似推理 (Rejection Sampling, N=500000) 结果: {approx_res[True]:.4f}")

if abs(exact_res[True] - approx_res[True]) < 0.05:
    print("\n所有测试通过！请保存并提交您的代码。")
else:
    print("\n注意：近似推理与精确推理的偏差较大，请检查 TODO 5 或增加 N 的数量。")


=== 最终测试 ===
查询: P(B | JohnCalls=True, MaryCalls=True)
精确推理 (Enumeration) 结果: 0.2842
近似推理 (Rejection Sampling, N=500000) 结果: 0.2616

所有测试通过！请保存并提交您的代码。
